### ***Sistem Rekomendasi Karakter Uma Musume Berdasarkan Karakteristik Gameplay Menggunakan Content-Based Filtering***

### Import Library

In [2]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics.pairwise import cosine_similarity

pd.set_option('display.max_columns', None)

### Load Dataset

In [5]:
df = pd.read_csv('/content/drive/MyDrive/Umamusume Dataset/data.csv')
df = df.rename(columns={'Middle': 'Medium'})

print("Jumlah baris (kostum):", df.shape[0])
print("Jumlah karakter unik:", df['Character'].nunique())
df.head()

Jumlah baris (kostum): 267
Jumlah karakter unik: 134


,Id,Name,Character,Release Date (JP),Release Date (EN),Rarity,Speed%,Stamina%,Power%,Guts%,Wit%,Turf,Dirt,Sprint,Mile,Medium,Long,Front,Pace,Late,End
0,0,Special Dreamer,Special Week,2021-03-02,2025-06-26,3,0,20,0,0,10,A,G,F,C,A,A,G,A,A,C
1,1,Hopp'n♪Happy Heart,Special Week,2021-07-29,2025-10-14,3,0,10,10,10,0,A,G,F,C,A,A,G,A,A,C
2,2,Ruler of Japan,Special Week,NaN,NaN,3,10,10,0,0,10,A,G,F,C,A,A,G,A,A,C
3,3,Innocent Silence,Silence Suzuka,2021-03-02,2025-06-26,3,20,0,0,10,0,A,G,D,A,A,E,A,C,E,G
4,4,Emerald Tidings,Silence Suzuka,2023-07-31,NaN,3,15,15,0,0,0,A,G,D,A,A,E,A,C,E,G


### EDA

In [6]:
df.info()
print("Jumlah nilai kosong per kolom:")
print(df.isnull().sum())

df[['Speed%','Stamina%','Power%','Guts%','Wit%']].describe()

aptitude_cols = ['Turf','Dirt','Sprint','Mile','Medium','Long','Front','Pace','Late','End']
for col in aptitude_cols:
    print(col, sorted(df[col].unique()))

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 267 entries, 0 to 266
Data columns (total 21 columns):
 #   Column             Non-Null Count  Dtype 
---  ------             --------------  ----- 
 0   Id                 267 non-null    int64 
 1   Name               267 non-null    object
 2   Character          267 non-null    object
 3   Release Date (JP)  265 non-null    object
 4   Release Date (EN)  102 non-null    object
 5   Rarity             267 non-null    int64 
 6   Speed%             267 non-null    int64 
 7   Stamina%           267 non-null    int64 
 8   Power%             267 non-null    int64 
 9   Guts%              267 non-null    int64 
 10  Wit%               267 non-null    int64 
 11  Turf               267 non-null    object
 12  Dirt               267 non-null    object
 13  Sprint             267 non-null    object
 14  Mile               267 non-null    object
 15  Medium             267 non-null    object
 16  Long               267 non-null    object
 1

### Encoding Aptitude

In [7]:
stat_cols = ['Speed%','Stamina%','Power%','Guts%','Wit%']

# Encoding ordinal aptitude: G (terendah) -> A (tertinggi)
grade_map = {'A':7, 'B':6, 'C':5, 'D':4, 'E':3, 'F':2, 'G':1}
for col in aptitude_cols:
    df[col + '_num'] = df[col].map(grade_map)

apt_num_cols = [c + '_num' for c in aptitude_cols]
df[apt_num_cols].head()

,Turf_num,Dirt_num,Sprint_num,Mile_num,Medium_num,Long_num,Front_num,Pace_num,Late_num,End_num
0,7,1,2,5,7,7,1,7,7,5
1,7,1,2,5,7,7,1,7,7,5
2,7,1,2,5,7,7,1,7,7,5
3,7,1,4,7,7,3,7,5,3,1
4,7,1,4,7,7,3,7,5,3,1


In [8]:
# Agregasi ke Level Karakter
agg_dict = {c: 'mean' for c in stat_cols}
agg_dict.update({c: 'max' for c in apt_num_cols})
agg_dict['Rarity'] = 'max'

char_df = df.groupby('Character').agg(agg_dict).reset_index()
print("Jumlah karakter setelah agregasi:", char_df.shape[0])
char_df.head()

Jumlah karakter setelah agregasi: 134


,Character,Speed%,Stamina%,Power%,Guts%,Wit%,Turf_num,Dirt_num,Sprint_num,Mile_num,Medium_num,Long_num,Front_num,Pace_num,Late_num,End_num,Rarity
0,Admire Groove,10.0,0.0,0.000000,10.000000,10.000000,7,1,1,6,7,1,1,6,7,5,3
1,Admire Vega,10.0,0.0,20.000000,0.000000,0.000000,7,1,2,5,7,5,1,1,6,7,3
2,Agnes Digital,7.5,7.5,7.500000,4.000000,3.500000,7,7,2,7,7,1,1,7,7,6,3
3,Agnes Tachyon,14.0,0.0,5.333333,3.333333,7.333333,7,1,1,4,7,6,3,7,6,2,3
4,Air Groove,10.0,0.0,15.000000,5.000000,0.000000,7,1,5,6,7,3,4,7,7,1,3


### Normalization Feature

In [9]:
feature_cols = stat_cols + apt_num_cols

scaler = MinMaxScaler()
X = scaler.fit_transform(char_df[feature_cols])

feature_df = pd.DataFrame(X, columns=feature_cols, index=char_df['Character'])
feature_df.head()

,Speed%,Stamina%,Power%,Guts%,Wit%,Turf_num,Dirt_num,Sprint_num,Mile_num,Medium_num,Long_num,Front_num,Pace_num,Late_num,End_num
Character,,,,,,,,,,,,,,,
Admire Groove,0.333333,0.000,0.000000,0.500000,0.465116,1.0,0.0,0.000000,0.833333,1.0,0.000000,0.000000,0.833333,1.000000,0.666667
Admire Vega,0.333333,0.000,0.800000,0.000000,0.000000,1.0,0.0,0.166667,0.666667,1.0,0.666667,0.000000,0.000000,0.833333,1.000000
Agnes Digital,0.250000,0.375,0.300000,0.200000,0.162791,1.0,1.0,0.166667,1.000000,1.0,0.000000,0.000000,1.000000,1.000000,0.833333
Agnes Tachyon,0.466667,0.000,0.213333,0.166667,0.341085,1.0,0.0,0.000000,0.500000,1.0,0.833333,0.333333,1.000000,0.833333,0.166667
Air Groove,0.333333,0.000,0.600000,0.250000,0.000000,1.0,0.0,0.666667,0.833333,1.0,0.333333,0.500000,1.000000,1.000000,0.000000


### Cosine Similarity

In [10]:
sim_matrix = cosine_similarity(X)
sim_df = pd.DataFrame(sim_matrix, index=char_df['Character'], columns=char_df['Character'])
sim_df.iloc[:5, :5]

Character,Admire Groove,Admire Vega,Agnes Digital,Agnes Tachyon,Air Groove
Character,,,,,
Admire Groove,1.000000,0.773547,0.885479,0.866938,0.833764
Admire Vega,0.773547,1.000000,0.760173,0.773043,0.758319
Agnes Digital,0.885479,0.760173,1.000000,0.782498,0.804341
Agnes Tachyon,0.866938,0.773043,0.782498,1.000000,0.897437
Air Groove,0.833764,0.758319,0.804341,0.897437,1.000000


In [11]:
def recommend(character_name, top_n=5):
    """
    Mengembalikan top_n karakter paling mirip dengan character_name
    berdasarkan cosine similarity dari fitur stat & aptitude.
    """
    if character_name not in char_df['Character'].values:
        return f"Karakter '{character_name}' tidak ditemukan dalam dataset."

    idx = char_df.index[char_df['Character'] == character_name][0]
    scores = list(enumerate(sim_matrix[idx]))
    scores = sorted(scores, key=lambda x: x[1], reverse=True)
    scores = [s for s in scores if s[0] != idx][:top_n]

    result = char_df.iloc[[s[0] for s in scores]][['Character']].copy()
    result['similarity_score'] = [round(s[1], 4) for s in scores]
    return result.reset_index(drop=True)

In [12]:
recommend('Special Week', top_n=5)
recommend('Gold Ship', top_n=5)

,Character,similarity_score
0,Rulership,0.9837
1,Marvelous Sunday,0.9740
2,Jungle Pocket,0.9725
3,Narita Top Road,0.9716
4,Tamamo Cross,0.9707


### Evaluation Precision@K

In [13]:
style_cols = ['Front_num','Pace_num','Late_num','End_num']
char_df['dominant_style'] = char_df[style_cols].idxmax(axis=1).str.replace('_num', '', regex=False)

def precision_at_k(character_name, k=5):
    if character_name not in char_df['Character'].values:
        return None
    query_style = char_df.loc[char_df['Character'] == character_name, 'dominant_style'].values[0]
    rec = recommend(character_name, top_n=k)
    rec_styles = char_df.set_index('Character').loc[rec['Character'], 'dominant_style']
    relevant = (rec_styles == query_style).sum()
    return relevant / k

precisions = [precision_at_k(c, k=5) for c in char_df['Character']]
print("Rata-rata Precision@5:", round(np.mean(precisions), 4))

Rata-rata Precision@5: 0.6239
